# Paragraph keystroke identification: classifier comparison

This is a **public, runnable demonstration** of the project's exploratory
classifier benchmark. It uses the **synthetic**
`sample_data/raw/aggregated_paragraph.csv` bundled with the repository.
These rows are examples of *aggregated timing features*, not actual
participants' key events or measurements. The private research notebook,
participant records, and fitted models should stay outside Git.

**What is compared:** 16 configured classifier choices, including three
SVM kernels. With only two sessions and one row per participant per session
in the example, LDA and QDA cannot be fitted; the table marks them as
skipped. Accuracy and macro F1 measure **closed-set identification**.
They are not verification FAR, FRR, or EER. This demonstration does not
reproduce historical percentages or evaluate the deployed model.

Run Jupyter from the repository root or `notebooks/`. Install the packages
in `requirements.txt` into your own virtual environment first.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis,
)
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (p / "manage.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run Jupyter from the repository root or notebooks/ directory.")
CSV_PATH = PROJECT_ROOT / "sample_data" / "raw" / "aggregated_paragraph.csv"
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Missing public example: {CSV_PATH}")
print("Found the public synthetic aggregated feature CSV.")


## 1. Read data and define the 501 real features

`subject` is the label. `sessionIndex` and `rep` describe collection and
must **not** enter the predictors. The previous private comparison included
numeric `rep`, making its feature count 502. Missing feature values in the
aggregated export are replaced with zero, matching the companion public RF
notebook and the project's current feature handling. No scaler is fitted
on all rows before the evaluation split.


In [ ]:
df = pd.read_csv(CSV_PATH)
required = {"subject", "sessionIndex", "rep"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(df.columns))}")
if df["subject"].isna().any() or df["subject"].astype(str).str.strip().eq("").any():
    raise ValueError("Every row needs a participant label.")

feature_columns = [
    c for c in df.columns
    if c not in {"subject", "participant", "subject_id", "sessionIndex", "rep"}
]
if not feature_columns:
    raise ValueError("No timing features found.")
X = df[feature_columns].apply(pd.to_numeric, errors="raise").fillna(0.0).to_numpy(dtype=float)
if not np.isfinite(X).all():
    raise ValueError("Non-finite values found in timing features.")
y = LabelEncoder().fit_transform(df["subject"].astype(str))
groups = pd.to_numeric(df["sessionIndex"], errors="raise").to_numpy()
if len(np.unique(groups)) != 2:
    raise ValueError("This demonstration expects exactly two collection sessions.")

folds = list(GroupKFold(n_splits=2).split(X, y, groups))
for train_index, test_index in folds:
    if set(y[train_index]) != set(y[test_index]):
        raise ValueError("Every class must occur in both train and test sessions.")
print(f"Rows: {len(y)} | people: {len(np.unique(y))} | timing features: {len(feature_columns)}")
print("Session held out in each fold:", [np.unique(groups[te]).tolist() for _, te in folds])


## 2. Configure 16 classifier choices

Every choice receives the same two session-held-out folds. `StandardScaler`
is inside the pipeline for methods that use it, so its statistics are fitted
**only on the training session** in each fold. No hyperparameter search is
performed here. Settings are illustrative and can differ from the earlier
research experiments or the model used by the Django app.


In [ ]:
RANDOM_STATE = 42
models = [
    ("RandomForest", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), False),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), False),
    ("SVM (Linear)", SVC(kernel="linear", class_weight="balanced"), True),
    ("SVM (RBF)", SVC(kernel="rbf", class_weight="balanced"), True),
    ("SVM (Poly)", SVC(kernel="poly", class_weight="balanced"), True),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=50, min_samples_leaf=2, random_state=RANDOM_STATE), False),
    ("LogisticRegression", LogisticRegression(max_iter=1000, class_weight="balanced"), True),
    ("KNN", KNeighborsClassifier(n_neighbors=5, weights="distance"), True),
    ("MLPClassifier", MLPClassifier(hidden_layer_sizes=(32,), max_iter=200, random_state=RANDOM_STATE), True),
    ("DecisionTree", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE), False),
    ("GradientBoosting", GradientBoostingClassifier(n_estimators=50, random_state=RANDOM_STATE), False),
    ("AdaBoost", AdaBoostClassifier(n_estimators=50, random_state=RANDOM_STATE), False),
    ("Bagging", BaggingClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1), False),
    ("GaussianNB", GaussianNB(), True),
    ("LDA", LinearDiscriminantAnalysis(), True),
    ("QDA", QuadraticDiscriminantAnalysis(reg_param=0.1), True),
]
assert len(models) == 16


## 3. Evaluate identification

Each test session is predicted once per classifier. The table reports the
**mean of the two fold scores**; status explains models that could not be
trained. With only one training observation per person in each fold, this
is a small educational benchmark, **not** a dependable model ranking.


In [ ]:
results = []
for name, estimator, scale in models:
    accuracies, macro_f1s = [], []
    status = "ok"
    for train_index, test_index in folds:
        counts = np.unique(y[train_index], return_counts=True)[1]
        if name == "LDA" and len(train_index) <= len(counts):
            status = "skipped: LDA requires more training rows than classes"
            break
        if name == "QDA" and counts.min() < 2:
            status = "skipped: QDA requires at least two training rows per class"
            break
        model = make_pipeline(StandardScaler(), estimator) if scale else estimator
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                model.fit(X[train_index], y[train_index])
            pred = model.predict(X[test_index])
        except (ValueError, FloatingPointError) as exc:
            status = f"failed: {type(exc).__name__}: {str(exc).splitlines()[0][:90]}"
            break
        accuracies.append(accuracy_score(y[test_index], pred))
        macro_f1s.append(f1_score(y[test_index], pred, average="macro", zero_division=0))
    if len(accuracies) != len(folds):
        accuracies, macro_f1s = [], []  # do not present one successful fold as a two-fold mean
    results.append({
        "Classifier": name,
        "Accuracy (%)": round(float(np.mean(accuracies)) * 100, 2) if accuracies else None,
        "Macro F1": round(float(np.mean(macro_f1s)), 4) if macro_f1s else None,
        "Status": status,
    })

comparison = pd.DataFrame(results)
display(comparison)
print(f"Evaluated: {(comparison['Status'] == 'ok').sum()} / {len(comparison)} classifiers")


## What these numbers cannot establish

The bundled data is fictional; **do not quote these scores as research
results**. The original private notebook's approximately 93% ExtraTrees
and 90% Random Forest figures used a different processed dataset and
feature selection, so these results are not a reproduction. Testing 16
choices on the same two sessions and then reporting the winning choice as
independently validated would also make the selected result optimistic.

Identification selects a class. **Verification** tests a claimed identity
using a class score and a decision threshold. The private notebook swept
thresholds on the evaluation session to report FAR/FRR/EER, then tried
thresholds obtained from the same records used to fit the classifier.
Neither procedure supplies a reliable, independent estimate here, so this
public notebook deliberately does **not** produce FAR/FRR/EER or threshold
files. For verification research, gather more independent sessions:
train the model, calibrate thresholds on separate enrollment/development
sessions, and report FAR/FRR on an untouched final session with both genuine
and impostor claims. Keep all participant data and fitted artifacts private.

The Django application's active paragraph model is a separately promoted
Random Forest. Running this notebook does not train or replace it.
